# Наскрізна демонстрація каскаду, Модель 1 на ORD

Той самий прогін, що й у підрозділі 3.8 пояснювальної записки, але першою ланкою
стоїть конфігурація на 147 000 реакціях ORD (канонічний запис), а не патентна A5.
Причина — конвенція лівої частини: підсумкова Модель 2 (M3) навчена на
`full_reactants_smiles`, тобто на повному завантаженні колби ORD, 3,97 молекули на
запис, тоді як патентна конфігурація видає лише субстрати, 1,71. ORD-конфігурація
пише реактантний бік так само, як його бачила Модель 2, тож стик каскаду перестає
міряти розбіжність джерел.

Три цільові молекули ті самі, що у v4 і v5, промені ті самі; Моделі 2 подається
найкращий уцілілий набір, бо за узгоджених конвенцій він уже має форму навчального
входу.

In [ ]:
import os
if not os.path.isdir("retro-planner"):
    !git clone https://github.com/oleh-kuzmenko/retro-planner.git
%cd retro-planner


In [ ]:
%pip install -q -e ".[local-models]"


In [ ]:
import glob, os, tarfile

# Модель 1 приходить виходом свого ядра, Модель 2 і три цільові записи — наборами даних.
model1_dir = next(glob.iglob("/kaggle/input/**/model1_compoundt5_150k/final", recursive=True))
test_file = next(glob.iglob("/kaggle/input/**/demo_targets.jsonl", recursive=True))

found = glob.glob("/kaggle/input/**/model2_m3_fullside/conditions_format.json", recursive=True)
if found:
    model2_dir = os.path.dirname(found[0])
else:
    # Набір даних лишився архівом — розпакувати у робочий каталог.
    archive = next(glob.iglob("/kaggle/input/**/model2_m3_fullside.tar", recursive=True))
    tarfile.open(archive).extractall("/kaggle/working/model2")
    model2_dir = next(glob.iglob("/kaggle/working/model2/**/conditions_format.json", recursive=True))
    model2_dir = os.path.dirname(model2_dir)

for path in (model1_dir, model2_dir, test_file):
    print(path)


In [ ]:
import json
from safetensors import safe_open

# Чекпоінт, чий config оголошує зв'язані ембедінги, тоді як lm_head.weight збережено
# окремим тензором, вантажиться без помилки й генерує сміття (RESULTS.md). Перевірити обидва.
for path in (model1_dir, model2_dir):
    config = json.load(open(f"{path}/config.json"))
    with safe_open(f"{path}/model.safetensors", "pt") as handle:
        keys = set(handle.keys())
    tied = config.get("tie_word_embeddings", True)
    assert not (tied and "lm_head.weight" in keys), f"{path}: config каже tied, а lm_head збережено окремо"
    print(path, "| tie:", tied, "| lm_head у вагах:", "lm_head.weight" in keys, "| vocab:", config["vocab_size"])


In [ ]:
!python scripts/models/demo_cascade_run.py \
    --model1-dir "{model1_dir}" --model2-dir "{model2_dir}" \
    --test-file "{test_file}" \
    --product "CCCCCCCCCCCCN1CCCC1" \
    --product "Nc1cc(NN2CCOCC2)ccc1[N+](=O)[O-]" \
    --product "C[C@H](Br)c1cc(Br)cc(C(F)(F)F)c1" \
    --num-beams 10 \
    --model2-input best \
    --output /kaggle/working/demo_cascade_v6.json


In [ ]:
import json

# Провенанс має писати сам сценарій; якщо клон репозиторію старіший за цю правку — дописати тут.
path = "/kaggle/working/demo_cascade_v6.json"
payload = json.load(open(path))
payload.setdefault("model1_dir", model1_dir)
payload.setdefault("model2_dir", model2_dir)
payload.setdefault("test_file", test_file)
json.dump(payload, open(path, "w"), ensure_ascii=False, indent=1)


In [ ]:
import json

payload = json.load(open("/kaggle/working/demo_cascade_v6.json"))
print("Модель 1:", payload["model1_dir"])
print("Модель 2:", payload["model2_dir"])
print("еталонний бік:", payload.get("reference_field"), "| вхід Моделі 2:", payload.get("model2_input_mode"))
for record in payload["records"]:
    print("\nпродукт   :", record["product_smiles"])
    print("  еталон  :", record["reference_substrates"])
    print("  прогноз :", record["predicted_substrates"])
    print("  умови   :", record.get("predicted_conditions", [{}])[0])
    print("  еталон  :", record["reference_conditions"])
